In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, row_number
from pyspark.sql.window import Window


# 1. INICIALIZAÇÃO DEFENSIVA (4 GB de RAM e 50 partições)
spark = SparkSession.builder \
    .appName("TCC_Gold_Modelagem") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "50") \
    .getOrCreate()

print("✅ SparkSession iniciada de forma segura com 4GB de RAM.")

# 2. Carrega a base selada da Silver
caminho_silver = "../data/processed/bacen_scr"
df_silver = spark.read.parquet(caminho_silver)

colunas_dimensao = [
    "uf", "segmento", "cliente", "cnae_ocupacao", 
    "porte", "modalidade", "submodalidade", "origem", "indexador"
]

print("\n=== CONSTRUINDO TABELAS DIMENSÃO (STAR SCHEMA) ===")

# 3. Loop inteligente para gerar todas as 9 dimensões
for coluna in colunas_dimensao:
    print(f"⏳ Gerando Dim_{coluna}...")
    df_dim = df_silver.select(coluna).distinct()
    window_spec = Window.orderBy(coluna)
    df_dim = df_dim.withColumn(f"id_{coluna}", row_number().over(window_spec))
    df_dim = df_dim.select(f"id_{coluna}", coluna)
    
    caminho_dim = f"../data/processed/gold/dim_{coluna}"
    df_dim.write.mode("overwrite").parquet(caminho_dim)

print("\n✅ Todas as 9 dimensões foram geradas e salvas!")

# 4. Gerando a dimensão de perfil de operação (adimplente/inadimplente) 
print("⏳ Gerando Dim_perfil_operacao...")
dados_perfil = [(0, "ADIMPLENTE"), (1, "INADIMPLENTE")]
df_dim_perfil = spark.createDataFrame(
    dados_perfil, ["id_perfil_operacao", "perfil_operacao"]
)
df_dim_perfil.write.mode("overwrite").parquet(
    "../data/processed/gold/dim_perfil_operacao"
)

print("\n✅ Todas as dimensões (incluindo o perfil) foram geradas e salvas!")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/09 11:58:45 WARN Utils: Your hostname, DESKTOP-P0CSUOC, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/11/09 11:58:45 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/haroldo/tcc_bacen_scr/.venv/lib/python3.14/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/11/09 11:58:47 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/09 11:58:48 WARN Utils: Service 'Spar

✅ SparkSession iniciada de forma segura com 4GB de RAM.



=== CONSTRUINDO TABELAS DIMENSÃO (STAR SCHEMA) ===
⏳ Gerando Dim_uf...


26/11/09 11:58:54 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 11:58:54 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 11:58:54 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 11:58:57 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 11:58:57 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 11:58:57 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 1

⏳ Gerando Dim_segmento...


26/11/09 11:58:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 11:58:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 11:58:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 11:58:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 11:59:00 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 11:59:00 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 1

... (12 outputs left out)

⏳ Gerando Dim_indexador...


26/11/09 11:59:07 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 11:59:07 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 11:59:07 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 11:59:07 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.



✅ Todas as 9 dimensões foram geradas e salvas!
⏳ Gerando Dim_perfil_operacao...



✅ Todas as dimensões (incluindo o perfil) foram geradas e salvas!


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.functions import broadcast

# 1. Carrega a base limpa da Silver
print("📥 Carregando base Silver limpa para a Tabela Fato...")
df_fato = spark.read.parquet("../data/processed/bacen_scr")

# 2. Aplicando a premissa metodológica do saldo zero (0 = Adimplente, 1 = Inadimplente)
print(
    "⚙️ Criando a coluna alvo (id_perfil_operacao) baseada na regra de saldo zero..."
)
df_fato = df_fato.withColumn(
    "id_perfil_operacao",
    F.when(
        (F.col("carteira_inadimplencia").isNull())
        | (F.col("carteira_inadimplencia") == 0),
        0,
    ).otherwise(1),
)

# 3. Lista de dimensões tradicionais (apenas as que precisam ser cruzadas por texto)
dimensoes = [
    "uf",
    "segmento",
    "cliente",
    "cnae_ocupacao",
    "porte",
    "modalidade",
    "submodalidade",
    "origem",
    "indexador",
]

print("\n=== CONSTRUINDO A TABELA FATO COM BROADCAST JOIN ===")

# 4. Substituição otimizada das dimensões textuais
for dim in dimensoes:
    print(f"🔄 Cruzando e substituindo texto por ID na dimensão: {dim}...")
    df_dim = spark.read.parquet(f"../data/processed/gold/dim_{dim}")
    df_fato = df_fato.join(broadcast(df_dim), on=dim, how="left").drop(dim)

# 5. Gravando a Tabela Fato em Parquet
caminho_fato = "../data/processed/gold/fato_bacen"
print(f"\n⏳ Salvando a Tabela Fato final em Parquet: {caminho_fato}...")

df_fato.write.mode("overwrite").parquet(caminho_fato)

print("✅ Tabela Fato gerada e salva com sucesso!")

# 6. Exibe a estrutura final
print("\n=== ESTRUTURA FINAL DA TABELA FATO ===")
df_fato.printSchema()

📥 Carregando base Silver limpa para a Tabela Fato...
⚙️ Criando a coluna alvo (id_perfil_operacao) baseada na regra de saldo zero...

=== CONSTRUINDO A TABELA FATO COM BROADCAST JOIN ===
🔄 Cruzando e substituindo texto por ID na dimensão: uf...
🔄 Cruzando e substituindo texto por ID na dimensão: segmento...
🔄 Cruzando e substituindo texto por ID na dimensão: cliente...
🔄 Cruzando e substituindo texto por ID na dimensão: cnae_ocupacao...
🔄 Cruzando e substituindo texto por ID na dimensão: porte...
🔄 Cruzando e substituindo texto por ID na dimensão: modalidade...
🔄 Cruzando e substituindo texto por ID na dimensão: submodalidade...
🔄 Cruzando e substituindo texto por ID na dimensão: origem...
🔄 Cruzando e substituindo texto por ID na dimensão: indexador...
🔄 Cruzando e substituindo texto por ID na dimensão: perfil_operacao...


AnalysisException: [UNRESOLVED_USING_COLUMN_FOR_JOIN] USING column `perfil_operacao` cannot be resolved on the left side of the join. The left-side columns: [`a_vencer_acima_de_5400_dias`, `a_vencer_ate_90_dias`, `a_vencer_de_1081_ate_1800_dias`, `a_vencer_de_1801_ate_5400_dias`, `a_vencer_de_361_ate_1080_dias`, `a_vencer_de_91_ate_360_dias`, `ativo_problematico`, `carteira_a_vencer`, `carteira_ativa`, `carteira_inadimplencia`, `carteira_vencida`, `data_base`, `id_cliente`, `id_cnae_ocupacao`, `id_indexador`, `id_modalidade`, `id_origem`, `id_perfil_operacao`, `id_porte`, `id_segmento`, `id_submodalidade`, `id_uf`, `numero_de_operacoes`, `vencido_acima_de_90_dias`, `vencido_de_15_ate_90_dias`]. SQLSTATE: 42703